In [1]:
import csv
import hashlib
from pathlib import Path

import numpy as np
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from threadpoolctl import threadpool_limits


DATA_DIR = Path.cwd()
if (DATA_DIR / "homework_3").is_dir():
    DATA_DIR = DATA_DIR / "homework_3"

SEED = 42


def load_wine(path):
    with path.open(encoding="utf-8-sig", newline="") as stream:
        header = stream.readline()
        stream.seek(0)
        reader = csv.DictReader(stream, delimiter=";" if ";" in header else ",")
        if not reader.fieldnames or reader.fieldnames[-1] != "quality":
            raise ValueError("The wine CSV must have a final column named 'quality'.")
        rows = np.array([[float(row[name]) for name in reader.fieldnames] for row in reader])
    return rows[:, :-1], rows[:, -1]

In [2]:
FOREST_TREES = (10, 20, 30, 40, 50, 60)
FOREST_GRID = {
    "max_depth": [3, 5, 7, 9, 12, 16, None],
    "max_features": [0.5, "sqrt", 1.0],
    "min_samples_leaf": [1, 2, 4, 8],
}
SCALES = {
    "standard": StandardScaler,
    "robust": RobustScaler,
    "minmax": MinMaxScaler,
}
NEIGHBOR_GRID = {
    "scaling": list(SCALES),
    "n_neighbors": list(range(1, 43)),
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}


def forest_candidates(x_train, y_train):
    for params in ParameterGrid(FOREST_GRID):
        forest = RandomForestRegressor(
            **params, random_state=SEED, warm_start=True, n_jobs=1
        )
        for trees in FOREST_TREES:
            forest.set_params(n_estimators=trees)
            forest.fit(x_train, y_train)
            yield {**params, "n_estimators": trees}, forest


def make_knn(params):
    settings = params.copy()
    scaler = SCALES[settings.pop("scaling")]()
    return Pipeline([("scale", scaler), ("knn", KNeighborsRegressor(**settings))])


def knn_candidates(x_train, y_train):
    for params in ParameterGrid(NEIGHBOR_GRID):
        yield params, make_knn(params).fit(x_train, y_train)


def select_by_mse(candidates, x_val, y_val):
    best_params = None
    best_mse = float("inf")
    count = 0
    for params, model in candidates:
        mse = mean_squared_error(y_val, model.predict(x_val))
        count += 1
        if mse < best_mse:
            best_params = params
            best_mse = mse
    if count <= len(y_val):
        raise ValueError(f"Grid of {count} must exceed {len(y_val)} validation rows.")
    return {"grid_size": count, "parameters": best_params, "validation_mse": best_mse}


def run_wine(path):
    x, y = load_wine(path)
    x_fit, x_test, y_fit, y_test = train_test_split(x, y, test_size=0.2, random_state=SEED)
    x_train, x_val, y_train, y_val = train_test_split(
        x_fit, y_fit, test_size=0.125, random_state=SEED
    )

    print("2(c): searching RandomForestRegressor...", flush=True)
    forest = select_by_mse(forest_candidates(x_train, y_train), x_val, y_val)
    print("2(c): searching KNeighborsRegressor...", flush=True)
    knn = select_by_mse(knn_candidates(x_train, y_train), x_val, y_val)

    fitted_forest = RandomForestRegressor(**forest["parameters"], random_state=SEED)
    fitted_knn = make_knn(knn["parameters"])
    for result, model in [(forest, fitted_forest), (knn, fitted_knn)]:
        model.fit(x_train, y_train)
        result["test_mse"] = mean_squared_error(y_test, model.predict(x_test))

    return {
        "file": path.name,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "sizes": {"train": len(y_train), "validation": len(y_val), "test": len(y_test)},
        "models": {"RandomForestRegressor": forest, "KNeighborsRegressor": knn},
    }

In [3]:
wine_path = DATA_DIR / "winequality-white.csv"
if not wine_path.is_file():
    wine_path = DATA_DIR / "winequality-white-uci.csv"

with threadpool_limits(limits=1):
    result = run_wine(wine_path)

sizes = result["sizes"]
print(f"Data: {result['file']} | train={sizes['train']:,}, validation={sizes['validation']:,}, test={sizes['test']:,}")
print(f"scikit-learn: {sklearn.__version__} | seed: {SEED}")
print(f"SHA-256: {result['sha256']}")
print(f"{'Model':<23}{'Grid':>6}{'Validation MSE':>18}{'Test MSE':>12}")
for name, metrics in result["models"].items():
    print(
        f"{name:<23}{metrics['grid_size']:>6}"
        f"{metrics['validation_mse']:>18.4f}{metrics['test_mse']:>12.4f}"
    )
for name, metrics in result["models"].items():
    print(f"{name}: {metrics['parameters']}")

2(c): searching RandomForestRegressor...
2(c): searching KNeighborsRegressor...
Data: winequality-white-uci.csv | train=3,428, validation=490, test=980
scikit-learn: 1.7.2 | seed: 42
SHA-256: 76c3f809815c17c07212622f776311faeb31e87610d52c26d87d6e361b169836
Model                    Grid    Validation MSE    Test MSE
RandomForestRegressor     504            0.3081      0.3553
KNeighborsRegressor       504            0.2987      0.3686
RandomForestRegressor: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 60}
KNeighborsRegressor: {'n_neighbors': 25, 'p': 1, 'scaling': 'standard', 'weights': 'distance'}
